# The `UnichartNotebook` Tutorial

A complete, hands-on guide to plotting with **`unichart`**.

`UnichartNotebook` is a thin, stateful wrapper around [Plotly](https://plotly.com/python/).
The core idea is simple:

> **Load** your data as *datasets* → **select** which datasets are active →
> **format** them (colors, markers, lines, axes, fonts) → **plot**.

Formatting is *remembered* on the notebook object, so you set it once and it
applies to every plot until you change it. This makes it fast to iterate:
tweak one thing, re-run the plot cell, repeat.

This tutorial focuses heavily on the two workhorse methods — **`plot()`** (scatter &
line charts) and **`plot_ymult()`** (multi-axis charts) — and the full set of
formatting options. Other chart types (`bar`, `box`, `histogram`, `contour`)
each get their own section.

---

### Table of contents

1. [Setup & sample data](#1)
2. [Loading datasets](#2)
3. [Selecting datasets](#3)
4. [**Scatter & line plots — `plot()`**](#4)  ← the main event
5. [**Multi-axis plots — `plot_ymult()`**](#5)
6. [Decorations & layout (lines, highlights, limits, labels, fonts, dark mode)](#6)
7. [Bar charts — `bar()`](#7)
8. [Box plots — `box()`](#8)
9. [Histograms — `histogram()`](#9)
10. [Contour plots — `contour()`](#10)
11. [**Regression & data tables** (`reg_order`, `reg_info`, `table`, `table_read`)](#11)
12. [Saving & inspecting](#12)
13. [Cheat sheet](#13)

<a id="1"></a>
## 1. Setup & sample data

`unichart` depends on `pandas`, `numpy`, `plotly`, `scipy`, and `ipywidgets`.
If you are running this notebook from the repository root, a plain
`import unichart` works. Otherwise, add the repo folder to `sys.path` first
(uncomment the lines below).

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

# import sys
# sys.path.append(r"/path/to/toms_utils")   # <- only if unichart isn't importable

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

Throughout the tutorial we'll use a small synthetic **jet-engine** dataset: three
engines (A, B, C) each logged over a 120-second run that idles, climbs, and
cruises. The columns are deliberately on very different scales (e.g. `EGT` in the
hundreds, `N1` as a percentage, `FF` fuel flow in the thousands) — perfect for
showing off multi-axis plots later.

In [2]:
def make_engine_data(seed=1):
    rng = np.random.default_rng(seed)
    frames = []
    for i, name in enumerate(['Engine A', 'Engine B', 'Engine C']):
        t   = np.linspace(0, 120, 80)                       # time, seconds
        tra = np.clip(20 + 80 * (1 - np.exp(-t / 30)), 0, 100)  # throttle %
        N1  = 25 + 0.70 * tra + i * 1.5 + rng.normal(0, 0.5, t.size)   # fan speed %
        EGT = 380 + 3.0 * tra + i * 15  + rng.normal(0, 6,   t.size)   # exhaust temp, degC
        FF  = 500 + 22  * tra + i * 120 + rng.normal(0, 40,  t.size)   # fuel flow, pph
        ALT = np.clip((t - 20) * 350, 0, 35000)             # altitude, ft
        MACH = np.clip((t - 20) * 0.006, 0, 0.80)
        PHASE = np.where(t < 20, 'Idle', np.where(ALT < 34000, 'Climb', 'Cruise'))
        frames.append(pd.DataFrame({
            'TIME': t, 'TRA': tra, 'N1': N1, 'EGT': EGT, 'FF': FF,
            'ALT': ALT, 'MACH': MACH, 'PHASE': PHASE,
            'SET': i, 'TITLE': name,
        }))
    return pd.concat(frames, ignore_index=True)

engine_df = make_engine_data()
engine_df.head()

,TIME,TRA,N1,EGT,FF,ALT,MACH,PHASE,SET,TITLE
0,0.000000,20.000000,39.172792,444.651943,975.982657,0.0,0.0,Idle,0,Engine A
1,1.518987,23.949794,42.175665,453.011180,1017.428942,0.0,0.0,Idle,0,Engine A
2,3.037975,27.704578,44.558423,453.328638,1084.326512,0.0,0.0,Idle,0,Engine A
3,4.556962,31.273978,46.240206,466.650956,1197.287965,0.0,0.0,Idle,0,Engine A
4,6.075949,34.667149,49.719682,489.304181,1290.683346,0.0,0.0,Idle,0,Engine A


<a id="2"></a>
## 2. Loading datasets

Create a notebook object, then load data into it. The most common entry point is
**`load_df()`**. When you pass `set_idx_column`, the frame is split into one
**dataset** per unique value of that column; `set_name_column` supplies each
dataset's title.

> A *dataset* is one logical series (here, one engine). Every dataset carries its
> own formatting state (color, marker, etc.) and can be selected or hidden
> independently.

In [3]:
nb = UnichartNotebook()
nb.load_df(engine_df, set_idx_column='SET', set_name_column='TITLE')

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


`list_sets()` shows what's loaded — the index, title, whether it's selected, its
shape, and any active query.

In [4]:
nb.list_sets()


Loaded Datasets:
Set    Title     Selected  Shape    Query?  
--------------------------------------------
Set 0  Engine A  ✓         80 x 10  None    
Set 1  Engine B  ✓         80 x 10  None    
Set 2  Engine C  ✓         80 x 10  None    


Other ways to get data in:

| Call | What it does |
|------|--------------|
| `nb.load_df(df, ...)` | Split / load one (or a list of) DataFrame(s). |
| `nb.load(source, ...)` | Same, but `source` may also be a **file path** (`.csv`, `.tsv`, `.xlsx`, `.json`, `.parquet`), a dict, or a numpy array. |
| `nb.load_clipboard()` | Read whatever is on the clipboard. |
| `nb.clear_data()` | Drop every dataset and start over. |

`load_df` auto-detects common column names too: if you don't pass
`set_idx_column`, it looks for `SETNUMBER`/`INDEX`, and for titles it looks for a
`TITLE` column.

<a id="3"></a>
## 3. Selecting datasets

Every plotting method only draws the **selected** datasets. By default all loaded
datasets are selected. The selection methods accept a flexible *slice*:

| Slice form | Meaning |
|------------|---------|
| `0` | a single dataset by index |
| `[0, 2]` | a list of datasets |
| `-1` | negative indexing (last dataset) |
| `'all'` | every dataset |
| `None` | the currently-selected datasets |

| Method | Effect |
|--------|--------|
| `nb.select(slice)` | Make *only* these datasets active (deselects the rest). |
| `nb.omit(slice)` | Hide these datasets. |
| `nb.restore(slice)` | Re-activate these (use `'all'` to bring everything back). |
| `nb.selected()` | Return the list of active `Dataset` objects. |

In [5]:
nb.select([0, 2])               # show only Engine A and Engine C
print("Active:", [d.title for d in nb.selected()])

nb.restore('all')               # bring everyone back
print("Active:", [d.title for d in nb.selected()])

Active: ['Engine A', 'Engine C']
Active: ['Engine A', 'Engine B', 'Engine C']


### Filtering rows with `query()`

`query()` applies a pandas-style boolean expression to the rows of the chosen
datasets. Pass an empty/no query string to clear it. (If a query removes every
row from a dataset, that dataset is automatically turned off.)

In [6]:
nb.query('all', 'PHASE == "Cruise"')   # keep only cruise rows
nb.list_sets()


Loaded Datasets:
Set    Title     Selected  Shape   Query?             
------------------------------------------------------
Set 0  Engine A  ✓         2 x 10  PHASE == "Cruise"  
Set 1  Engine B  ✓         2 x 10  PHASE == "Cruise"  
Set 2  Engine C  ✓         2 x 10  PHASE == "Cruise"  


In [7]:
nb.query('all')   # clear the query (restores all rows)
nb.list_sets()


Loaded Datasets:
Set    Title     Selected  Shape    Query?  
--------------------------------------------
Set 0  Engine A  ✓         80 x 10  None    
Set 1  Engine B  ✓         80 x 10  None    
Set 2  Engine C  ✓         80 x 10  None    


<a id="4"></a>
# 4. Scatter & line plots — `plot()`

`plot()` is the primary method. Its signature:

```python
nb.plot(x=None, y=None, by='vars', figsize=(12, 8),
        ncols=None, nrows=None, subplot_titles=None,
        suptitle=None, suppress_legends=False, **kwargs)
```

Key ideas:

- **`x` / `y`** are column names (or lists of them). If omitted, the *last* `x`/`y`
  you used are reused — handy for re-plotting after a formatting tweak.
- **`by`** controls the subplot layout:
  - `'vars'` *(default)* — one subplot per **y variable**.
  - `'sets'` / `'datasets'` — one subplot per **dataset**.
  - `'ymult'` — a single plot with multiple y-axes (delegates to `plot_ymult()`).
- Extra `**kwargs` flow through to the underlying engine (e.g. `legend`,
  `legend_ncols`, `grid`).

Every plotting method returns the Plotly `Figure`, so you can keep customizing it
or call `.show()`.

<a id="4.1"></a>
## 4.1 Your first plot

By default, datasets are drawn as **markers** (a scatter plot). Each dataset gets
a color from the palette automatically.

In [8]:
nb.plot('TIME', 'EGT').show()

<a id="4.2"></a>
## 4.2 Lines vs. markers — `linestyle`

The marker-vs-line decision is driven by **`linestyle`**. With no line style set,
you get markers. Set a line style and the trace becomes a line. `unichart`
accepts both Matplotlib-style and Plotly-style strings:

| Matplotlib | Plotly | Look |
|:---:|:---:|---|
| `'-'`  | `'solid'`   | solid line |
| `'--'` | `'dash'`    | dashed |
| `'-.'` | `'dashdot'` | dash-dot |
| `':'`  | `'dot'`     | dotted |

Apply it to a dataset slice with **`nb.linestyle(slice, style)`**.

In [9]:
nb.linestyle('all', '-')        # solid lines for every dataset
nb.plot('TIME', 'EGT').show()

You can mix: give each engine a different style. (We'll cover the per-dataset
formatting methods in depth in §4.6.)

In [10]:
nb.linestyle(0, '-')
nb.linestyle(1, '--')
nb.linestyle(2, ':')
nb.plot('TIME', 'EGT').show()

<a id="4.3"></a>
## 4.3 Multiple y-variables & the subplot grid

Pass a **list** for `y` and (with the default `by='vars'`) you get one subplot per
variable. The grid is chosen automatically, but you can pin it with `ncols` /
`nrows`.

In [11]:
nb.linestyle('all', '-')
nb.plot('TIME', ['EGT', 'N1', 'FF']).show()

In [12]:
# Force a single column so the three subplots stack vertically
nb.plot('TIME', ['EGT', 'N1', 'FF'], ncols=1, figsize=(9, 11)).show()

You can also give custom `subplot_titles` and an overall `suptitle`:

In [13]:
nb.plot('TIME', ['EGT', 'N1'],
        subplot_titles=['Exhaust Gas Temp', 'Fan Speed'],
        suptitle='Engine run — temperature & speed').show()

<a id="4.4"></a>
## 4.4 `by='sets'` — one subplot per dataset

Sometimes you want each *engine* in its own panel rather than each *variable*.
Switch `by` to `'sets'` (or the alias `'datasets'`). Here every panel shows the
same `x`/`y`, one per active dataset.

In [14]:
nb.plot('TIME', 'EGT', by='sets').show()

This combines naturally with selection — omit a dataset and its panel disappears.

In [15]:
nb.select([0, 1])                       # only A and B
nb.plot('TIME', 'EGT', by='sets').show()
nb.restore('all')

<a id="4.5"></a>
## 4.5 Pairing `x` and `y`

`x` and `y` can both be lists. The pairing rules:

- **Equal lengths** → zipped element-wise: `x=[a, b], y=[p, q]` → `(a,p), (b,q)`.
- **One `x`, many `y`** → that `x` is reused for every `y` (the common case).
- **Many `x`, one `y`** → that `y` is reused for every `x`.

In [16]:
# Two independent (x, y) pairs, zipped element-wise
nb.plot(x=['TIME', 'N1'], y=['EGT', 'FF'],
        subplot_titles=['EGT vs TIME', 'FF vs N1']).show()

<a id="4.6"></a>
## 4.6 Per-dataset formatting

This is the heart of `unichart`. Each method below takes a **dataset slice** and a
value, and the setting *persists* until you change it. Re-run your `plot()` cell
to see the effect.

| Method | Sets | Accepts |
|--------|------|---------|
| `nb.color(slice, val)` | line/marker color | `'red'`, `'#FF5733'`, `'rgb(0,128,0)'`, or an **int** (palette index) |
| `nb.marker(slice, val)` | marker symbol | `'o' 's' '^' 'D' '*' '.'` … or Plotly names (`'circle'`, `'star'`) |
| `nb.linestyle(slice, val)` | line dash | `'-' '--' '-.' ':'` or `'solid' 'dash' …` |
| `nb.markersize(slice, val)` | marker size | number |
| `nb.linewidth(slice, val)` | line thickness | number |
| `nb.alpha(slice, val)` | opacity | `0`–`1` |
| `nb.fill(slice, val)` | solid vs hollow markers | `True`/`False` (or `'on'`/`'off'`) |
| `nb.edgewidth(slice, val)` | marker outline thickness | number ≥ 0 |

Let's style all three engines distinctly.

In [17]:
nb.color(0, 'crimson')
nb.color(1, 'royalblue')
nb.color(2, 'seagreen')

nb.marker(0, 'o')
nb.marker(1, 's')
nb.marker(2, '^')

nb.markersize('all', 9)
nb.linestyle('all', '-')

nb.plot('TIME', 'EGT').show()

**Hollow markers** are great for dense scatter plots — `fill(slice, False)` keeps
just the outline. `edgewidth` controls how thick that outline is.

In [18]:
nb.fill('all', False)
nb.edgewidth('all', 2)
nb.linestyle('all', None)        # markers only, so the hollow shapes are visible
nb.plot('TIME', 'EGT').show()

In [19]:
nb.fill('all', True)             # back to solid for the rest of the tutorial
nb.linestyle('all', '-')

### The `color`/`marker` integer trick

Passing an **integer** to `color()` or `marker()` resolves to whatever the palette
would assign to *that* set index — independent of any recoloring. So
`nb.color(5, 2)` gives set 5 the palette color that set 2 started with.

In [20]:
nb.color('all', 0)               # give every engine the *first* palette color
nb.plot('TIME', 'EGT').show()

<a id="4.7"></a>
## 4.7 Color & marker maps (palettes)

New datasets pull their default color/marker from two ordered lists you can
replace wholesale:

- `nb.color_map` — list of colors assigned by dataset index (defaults to Plotly's
  qualitative palette).
- `nb.marker_map` — list of marker symbols, the marker analogue.

Both **cycle**, so a short list still answers any index. There's also
`nb.set_color_palette(palette, slice)` to recolor existing datasets from a named
or list palette.

In [21]:
import plotly.express as px

nb.color_map = px.colors.qualitative.Bold     # swap the default palette
nb.set_color_palette('Bold', 'all')           # recolor the current datasets too
nb.marker('all', 'o')
nb.plot('TIME', ['EGT', 'N1']).show()

<a id="4.8"></a>
## 4.8 Per-*variable* formatting — `var_format`

So far formatting has been *per dataset*. But often a **variable** has a canonical
look you want everywhere it appears — e.g. "`EGT` is always a red dashed line,
whatever engine it belongs to." That's what `var_format` is for:

```python
nb.var_format(variable, color=, marker=, linestyle=, markersize=, linewidth=, alpha=)
```

Variable formatting **overrides** dataset formatting on a *per-attribute* basis —
anything you don't set still falls back to the dataset's value. Pass `'reset'` to
clear one attribute.

There's a convenient shortcut: the per-dataset methods (`color`, `marker`,
`linestyle`, …) **auto-detect** when their first argument is a *variable name*
rather than a dataset index, and route to `var_format` for you. So
`nb.color('EGT', 'red')` styles the `EGT` variable, while `nb.color(0, 'red')`
styles dataset 0.

In [22]:
nb.reset_format()                       # clear earlier per-dataset styling
nb.color('EGT', 'firebrick')            # the EGT *variable* -> red everywhere
nb.linestyle('EGT', '--')
nb.color('N1', 'navy')                  # the N1 variable -> blue
nb.plot('TIME', ['EGT', 'N1'], by='sets').show()

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


In [23]:
nb.list_var_formats()      # inspect the active per-variable overrides

Variable-level formatting (overrides dataset attributes):
  EGT: color='firebrick', linestyle='--'
  N1: color='navy'


In [24]:
nb.clear_var_format()      # wipe all variable overrides before moving on

<a id="4.9"></a>
## 4.9 Coloring by a continuous variable — `hue`

Instead of one flat color per dataset, you can map a **numeric column** to a color
scale. Set `nb.hue(slice, column)` and (optionally) `nb.hue_palette(slice, scale)`.
A shared colorbar is added automatically.

In [25]:
nb.reset_format()
nb.marker('all', 'o')
nb.hue('all', 'ALT')                # color points by altitude
nb.hue_palette('all', 'Viridis')
nb.plot('TIME', 'EGT').show()

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


In [26]:
nb.hue('all', '')                   # turn hue back off

<a id="4.10"></a>
## 4.10 Regression & trendlines — `reg_order`

`nb.reg_order(slice, spec)` overlays a fitted trend on the scatter. The `spec`
accepts many forms:

| `spec` | Fit |
|--------|-----|
| `1`, `'linear'` | straight line |
| `2`, `'quadratic'`, `'poly2'` | quadratic (any `'polyN'`) |
| `'log'`, `'exp'`, `'power'` | log / exponential / power law |
| `'lowess'` | locally-weighted smoothing |
| `'spline'` | cubic spline |
| `'ma'`, `'rolling'` | moving average |

Use `nb.reg_info()` to print the fitted coefficients/R².

In [27]:
nb.reset_format()
nb.marker('all', 'o')
nb.markersize('all', 6)
nb.alpha('all', 0.5)
nb.reg_order('all', 'linear')       # add a linear trendline to each engine
nb.plot('TIME', 'EGT').show()

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


In [28]:
nb.reg_order('all', None)           # remove the trendlines
nb.alpha('all', 1)

<a id="4.11"></a>
## 4.11 Sorting & hover details

**Line ordering.** A line connects points in row order. To connect them in the
order of some column instead, set the dataset's `order` attribute (use
`'index'` to sort by the row index):

```python
nb.sets[0].order = 'TRA'      # connect this dataset's line in TRA order
```

**Hover tooltips.** Extra columns can be surfaced on hover. Set them globally via
`nb.display_parms`, or per dataset with `nb.set_display_parms(slice, cols)`.

In [29]:
nb.display_parms = ['PHASE', 'ALT']     # show phase & altitude on hover
nb.linestyle('all', '-')
nb.plot('TIME', 'EGT').show()
nb.display_parms = []

<a id="5"></a>
# 5. Multi-axis plots — `plot_ymult()`

```python
nb.plot_ymult(x=None, y=None, suptitle=None, figsize=(12, 8),
              legend='above', legend_group_by='sets', suppress_legends=False)
```

`plot_ymult` overlays **multiple y-variables on a single shared x-axis**, giving
each variable its **own y-axis**. This is the right tool when variables live on
wildly different scales — exactly our `EGT` (hundreds) vs `N1` (tens) vs `FF`
(thousands) situation.

(You can also reach it through the main method as `nb.plot(..., by='ymult')`.)

<a id="5.1"></a>
## 5.1 Why it's needed

Plot `EGT` and `FF` together on a *normal* shared axis and the small-scale series
gets crushed flat against the bottom:

In [30]:
nb.reset_format()
nb.marker('all', 'o'); nb.linestyle('all', '-')
nb.select(0)                                  # one engine, to keep it readable
nb.plot('TIME', ['EGT', 'FF']).show()         # two subplots, separate scales

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


<a id="5.2"></a>
## 5.2 Two y-axes on one plot

`plot_ymult` puts the **first** variable on the left axis and the **second** on the
right, so both are legible at their natural scale.

In [31]:
nb.plot_ymult('TIME', ['EGT', 'FF']).show()

<a id="5.3"></a>
## 5.3 Three or more axes

A third (and beyond) variable gets an additional axis stacked to the right; the
plot area shrinks automatically to make room.

In [32]:
nb.plot_ymult('TIME', ['EGT', 'N1', 'FF']).show()

<a id="5.4"></a>
## 5.4 Styling each axis — `var_format` precedence

Because every line on a multi-axis plot is a *different variable*, the natural way
to style them is **per variable** with `var_format` (§4.8). The resolver is
per-attribute: a variable override wins, otherwise the value falls back to the
dataset.

When a variable's `color` is set, its **y-axis label is tinted to match** — so you
can tell at a glance which axis belongs to which trace.

In [33]:
nb.var_format('EGT', color='firebrick', linestyle='-')
nb.var_format('N1',  color='seagreen',  linestyle='--')
nb.var_format('FF',  color='darkorange', linestyle=':')
nb.plot_ymult('TIME', ['EGT', 'N1', 'FF']).show()

With multiple datasets selected you get one trace per *(dataset × variable)*. The
variable controls the style; the dataset still distinguishes individual lines.

In [34]:
nb.restore('all')                  # all three engines
nb.plot_ymult('TIME', ['EGT', 'FF']).show()

<a id="5.5"></a>
## 5.5 Legends & grouping

- `legend` — `'above'` (default), `'right'`, or `'off'`.
- `legend_group_by` — `'sets'` or `'vars'`, controls how legend entries are grouped.
- `suppress_legends=True` — start every trace hidden (click legend items to reveal).

In [35]:
nb.plot_ymult('TIME', ['EGT', 'FF'],
              legend='right', legend_group_by='vars',
              suptitle='Grouped by variable').show()
nb.clear_var_format()

<a id="6"></a>
# 6. Decorations & layout

These notebook-level settings apply to **whatever you plot next** (and most chart
types, not just `plot`). Like formatting, they persist until changed.

## 6.1 Reference lines — `line()`

`nb.line(column, level, color='red', linestyle=None)` draws a line keyed to a
variable:

- keyed to the **x-variable** → a **vertical** line,
- keyed to a **y-variable** → a **horizontal** line.

Clear with `nb.line(column, 'clear')`, or `nb.line('all', 'clear')` for all.

In [36]:
nb.reset_format(); nb.marker('all', 'o'); nb.linestyle('all', '-')
nb.select(0)

nb.line('EGT', 650, color='red', linestyle='--')      # horizontal limit line
nb.line('TIME', 20, color='gray', linestyle=':')       # vertical event marker
nb.plot('TIME', 'EGT').show()

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


## 6.2 Highlighted regions — `highlight()`

`nb.highlight(column, (lo, hi), color='yellow', alpha=0.2)` shades a band. Keyed to
x → a vertical band; keyed to y → a horizontal band. Clear the same way as lines.

In [37]:
nb.highlight('TIME', (0, 20), color='gray', alpha=0.15)   # shade the idle phase
nb.highlight('EGT', (600, 650), color='orange', alpha=0.2)
nb.plot('TIME', 'EGT').show()

In [38]:
nb.line('all', 'clear')
nb.highlight('all', 'clear')

## 6.3 Axis limits — `scale()`

`nb.scale(column, (min, max))` pins an axis range; it follows the variable onto
whichever subplot uses it. Pass a list of columns to set several at once, and
`'clear'` to remove.

In [39]:
nb.scale('EGT', (400, 700))
nb.scale('TIME', (0, 120))
nb.plot('TIME', 'EGT').show()
nb.scale('EGT', 'clear'); nb.scale('TIME', 'clear')

Limits set for 'EGT': (400, 700)
Limits set for 'TIME': (0, 120)


Limits cleared for 'EGT'.
Limits cleared for 'TIME'.


## 6.4 Titles & axis labels

Set these attributes directly on the notebook; they feed the next plot:

- `nb.suptitle` — overall figure title
- `nb.x_label`, `nb.y_label` — axis labels
- `nb.set_title(slice, title)` — rename a dataset (changes its legend entry)

In [40]:
nb.suptitle = 'Engine A — Exhaust Gas Temperature'
nb.x_label = 'Time (s)'
nb.y_label = 'EGT (degC)'
nb.plot('TIME', 'EGT').show()
nb.suptitle = None; nb.x_label = None; nb.y_label = None

## 6.5 Font sizes — `set_font_sizes()`

One call controls the size of every text element. Sizes may be numbers **or named
sizes** (`'xs' 'sm' 'md' 'lg' 'xl' 'xxl' '3xl' 'huge'` …). `all=` sets a baseline
for everything; individual arguments override it. `reset=True` restores defaults.

In [41]:
nb.set_font_sizes(suptitle='xxl', axes_title='lg', legend='sm', axes_tick='sm')
nb.suptitle = 'Big title, small ticks'
nb.plot('TIME', ['EGT', 'N1']).show()
nb.suptitle = None
print(nb.get_font_sizes())
nb.set_font_sizes(reset=True)

{'suptitle': 24.0, 'legend': 10.0, 'axes_title': 14.0, 'axes_tick': 10.0, 'subplot_title': None, 'colorbar': None, 'hover': None, 'table_header': None, 'table_cell': None}


## 6.6 Dark mode — `toggle_darkmode()`

Flip every subsequent plot between light and dark themes. Pass an explicit
`state=True/False`, or call with no argument to toggle.

In [42]:
nb.toggle_darkmode(True)
nb.restore('all')
nb.plot('TIME', 'EGT', by='sets').show()
nb.toggle_darkmode(False)

Plot theme set to: Dark Mode


Plot theme set to: Light Mode


<a id="7"></a>
# 7. Bar charts — `bar()`

```python
nb.bar(x=None, y=None, markers=None, by='vars', barmode='group',
       agg='mean', figsize=(12, 8), ncols=None, nrows=None, suppress_legends=False)
```

Bars aggregate a numeric `y` over a categorical `x`. `agg` chooses the reducer
(`'mean'`, `'sum'`, …) and `barmode` is `'group'` or `'stack'`. Our categorical
column is `PHASE`.

- `by='vars'` *(default)* — a subplot per y-variable, bars colored by dataset.
- `by='sets'` — a subplot per dataset.
- `by='dataset_x'` — datasets themselves become the x-axis categories.

In [43]:
nb.reset_format()
nb.bar('PHASE', 'EGT', agg='mean').show()

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


In [44]:
nb.bar('PHASE', ['EGT', 'FF'], barmode='group', agg='mean').show()

**Overlay markers.** The `markers=` argument overlays a per-category value (styled
through `var_format`) on top of the bars — handy for drawing a limit or target on
each bar.

In [45]:
# add a per-row "EGT limit" column, then overlay its mean per phase
nb.add_column('EGT_LIMIT', 680)
nb.var_format('EGT_LIMIT', color='red', marker='*', markersize=18)
nb.bar('PHASE', 'EGT', markers='EGT_LIMIT', agg='mean').show()
nb.clear_var_format()

<a id="8"></a>
# 8. Box plots — `box()`

```python
nb.box(x=None, y=None, by='vars', boxmode='group', points='outliers',
       notched=False, color=None, suptitle=None, figsize=(12, 8),
       ncols=None, nrows=None, suppress_legends=False)
```

Box plots show the distribution of `y` for each category of `x`.

- `points` — `'outliers'` (default), `'all'`, `'suspectedoutliers'`, or `False`.
- `notched=True` — draw confidence notches.
- `boxmode` — `'group'` or `'overlay'`.
- `by='dataset_x'` — put one box per dataset on the x-axis.

In [46]:
nb.box('PHASE', 'EGT').show()

In [47]:
nb.box('PHASE', 'EGT', points='all', notched=True).show()

<a id="9"></a>
# 9. Histograms — `histogram()`

```python
nb.histogram(x=None, y=None, histfunc='sum', by='vars', nbins=None,
             bin_size=None, bin_start=None, bin_end=None, histnorm='',
             barmode='overlay', alpha=0.7, color=None, ...)
```

Histograms bin a single column (`x`). Control the bins with `nbins` *or* the
explicit `bin_size`/`bin_start`/`bin_end` trio. `histnorm` can normalize
(`'percent'`, `'probability'`, `'density'`, …), and `barmode='overlay'` (with
`alpha`) lets dataset distributions show through each other.

In [48]:
nb.restore('all')
nb.histogram('EGT', alpha=0.6).show()

In [49]:
nb.histogram('EGT', bin_size=25, histnorm='probability', barmode='overlay').show()

<a id="10"></a>
# 10. Contour plots — `contour()`

```python
nb.contour(x=None, y=None, z=None, by='vars', contours_coloring='fill',
           colorscale=None, interpolate=True, interp_res=100,
           interp_method='linear', ncontours=None, ...)
```

A contour maps a third variable `z` to color over the `x`–`y` plane. Scattered
data is interpolated onto a grid (`interpolate=True`). `contours_coloring` may be
`'fill'`, `'heatmap'`, or `'lines'`.

Contours need genuinely 2-D scattered data, so here's a small **engine map**:
`EGT` as a function of `MACH` and `ALT`.

In [50]:
def make_engine_map(seed=2):
    rng = np.random.default_rng(seed)
    frames = []
    for i, name in enumerate(['Build 1', 'Build 2']):
        n = 150
        mach = rng.uniform(0, 0.85, n)
        alt  = rng.uniform(0, 40000, n)
        egt  = 420 + 0.004 * alt + 320 * mach + rng.normal(0, 8, n) + i * 25
        frames.append(pd.DataFrame({'MACH': mach, 'ALT': alt, 'EGT': egt,
                                    'SET': i, 'TITLE': name}))
    return pd.concat(frames, ignore_index=True)

cmap = UnichartNotebook()
cmap.load_df(make_engine_map(), set_idx_column='SET', set_name_column='TITLE')
cmap.select(0)
cmap.contour('MACH', 'ALT', z='EGT').show()

UniChart Notebook Environment Initialized.
Loaded Set 0: Build 1
Loaded Set 1: Build 2


In [51]:
cmap.contour('MACH', 'ALT', z='EGT', contours_coloring='lines', ncontours=12).show()

<a id="11"></a>
# 11. Regression & data tables

`unichart` can fit **trend curves** to your scatter data and pull **tabular /
interpolated values** off them. Four tools work together:

| Tool | Role |
|------|------|
| `nb.reg_order(slice, spec)` | Attach a fitted curve to dataset(s) — drawn on the next scatter plot. |
| `nb.reg_info(slice)` | Print each curve's **equation** and **fit quality** (R², RMSE, MAE). |
| `nb.table(cols, x_in=…)` | Render a **values table** — raw rows, or values read off the fitted curve. |
| `nb.table_read(slice, …)` | Same idea, but **returns numbers** for use in code. |

`reg_order` was introduced briefly in [§4.10](#4.10); this section is the full
treatment, then builds on it with `reg_info`, `table`, and `table_read`.

## 11.1 `reg_order` — fitting a trend curve

`nb.reg_order(slice, spec)` attaches a regression to the chosen dataset(s); the
fitted curve is drawn the next time you scatter-plot that `x`/`y`. The `spec` is
flexible:

| `spec` | Fit | Notes |
|--------|-----|-------|
| `1`, `'linear'`, `'lin'` | straight line | |
| `2`, `'quadratic'`, `'poly2'` … | polynomial | any `'polyN'` or integer degree |
| `3`, `'cubic'` | cubic polynomial | |
| `'log'` | `y = a·ln(x) + b` | needs `x > 0` |
| `'exp'` | `y = A·e^(bx)` | needs `y > 0` |
| `'power'` | `y = A·x^b` | needs `x, y > 0` |
| `'lowess'`, `'loess'` | locally-weighted smoothing | requires `statsmodels`; default frac `0.3` |
| `'spline'`, `'cubic_spline'` | smoothing spline | default order `3` |
| `'ma'`, `'rolling'`, `'moving_average'` | centered moving average | |
| `(kind, param)` | any of the above with an explicit parameter | e.g. `('ma', 5)`, `('lowess', 0.5)` |
| `None` / `0` | clears the fit | |

Each dataset carries its own spec, so different engines can get different fits.

In [52]:
nb.restore('all')
nb.reset_format()
nb.marker('all', 'o')
nb.markersize('all', 6)
nb.alpha('all', 0.5)

# A different fit per engine, to show the variety
nb.reg_order(0, 'linear')
nb.reg_order(1, 'poly2')
nb.reg_order(2, ('ma', 7))

nb.plot('TIME', 'EGT').show()

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


## 11.2 `reg_info` — equations & fit statistics

Once a fit is set, `nb.reg_info(slice)` reports it. For each dataset it prints the
**fit type**, the **equation** (for the closed-form fits: poly / log / exp /
power), and goodness-of-fit stats — **R²**, **RMSE**, **MAE**, and the sample
count **n**.

> ⚠️ `reg_info` describes the fit for the **most recently plotted** `x` / `y`
> (`nb.last_x` / `nb.last_y`). Run a `plot()` first so it knows which columns to
> use.

It also **returns a dict** keyed by dataset index, so you can use the numbers
programmatically.

In [53]:
info = nb.reg_info('all')

Regression info for y='EGT' vs x='TIME':
Set 0 (Engine A) [y='EGT' vs x='TIME']: Linear (degree 1)  →  y = 1.578x + 524.9
   R²=0.7896   RMSE=28.57   MAE=23.25   n=80
Set 1 (Engine B) [y='EGT' vs x='TIME']: Polynomial (degree 2)  →  y = -0.02461x² + 4.568x + 480.1
   R²=0.9746   RMSE=10.13   MAE=8.176   n=80
Set 2 (Engine C) [y='EGT' vs x='TIME']: Moving average (window=7)  →  Moving average (window=7)
   R²=0.9890   RMSE=6.595   MAE=5   n=80


The returned dict carries the raw spec, parsed `kind`/`param`, human label,
`formula` string, and a `stats` sub-dict — handy for asserting fit quality in code:

In [54]:
for idx, d in info.items():
    r2 = d['stats']['r2'] if d['stats'] else None
    print(f"Set {idx}: {d['label']:<22}  R² = {r2:.4f}" if r2 is not None
          else f"Set {idx}: {d['label']}  (no closed-form stats)")

Set 0: Linear (degree 1)       R² = 0.7896
Set 1: Polynomial (degree 2)   R² = 0.9746
Set 2: Moving average (window=7)  R² = 0.9890


## 11.3 `table` — values as a table

`nb.table(cols=None, title=None, x_col=None, x_in=None, kind=None, output=None)`
renders a nicely-styled table of values from the selected datasets. It has two
modes.

**Raw mode** (no `x_in`): show the actual rows for `cols`. If you omit `cols`, it
falls back to the last plotted `x`/`y`.

In [55]:
nb.select(0)                                  # one engine keeps the table short
nb.table(['TIME', 'EGT', 'PHASE'], title='Engine A — raw sample')

Set,Dataset,TIME,EGT,PHASE
0,Engine A,0,444.65,Idle
0,Engine A,1.519,453.01,Idle
0,Engine A,3.038,453.33,Idle
0,Engine A,4.557,466.65,Idle
0,Engine A,6.0759,489.3,Idle
0,Engine A,7.5949,497.76,Idle
0,Engine A,9.1139,499.04,Idle
0,Engine A,10.633,511.62,Idle
0,Engine A,12.152,522.61,Idle
0,Engine A,13.671,530.65,Idle


**Interpolation / fit mode** (pass `x_in`): for each requested `x` value, the `y`
column(s) are read **off the fitted curve**. The regression spec comes from `kind`
if given, otherwise each dataset's own `reg_order`; `kind` accepts the *same*
specs as `reg_order` (`'poly2'`, `'log'`, `'exp'`, `('ma', 5)`, …), so the table
matches the plotted curve. When no spec is set, values are interpolated
piecewise-linearly through the raw points.

Two extra columns are added:

- **`INTERPOLATED`** — `True` when the value came from the model/interpolation
  rather than an exact raw point.
- **`METHOD`** — *how* it was produced: the regression label (`'Linear'`, `'LS2'`,
  `'Log'`, …) for a fitted curve, `'Table'` for piecewise-linear table lookup, or
  `None` for an exact point.

Here we read EGT at three throttle times using a quadratic fit, across all engines:

In [56]:
nb.restore('all')
nb.table(cols='EGT', x_col='TIME', x_in=[30, 60, 90], kind='poly2',
         title='EGT read off a quadratic fit at TIME = 30, 60, 90 s')

Set,Dataset,TIME,EGT,INTERPOLATED,METHOD
0,Engine A,30,580.19,Interpolated,LS2
0,Engine A,60,649.29,Interpolated,LS2
0,Engine A,90,674.86,Interpolated,LS2
1,Engine B,30,594.97,Interpolated,LS2
1,Engine B,60,665.56,Interpolated,LS2
1,Engine B,90,691.86,Interpolated,LS2
2,Engine C,30,609.4,Interpolated,LS2
2,Engine C,60,678.56,Interpolated,LS2
2,Engine C,90,706.14,Interpolated,LS2


**Output modes** — `output` controls what you get back instead of a rendered table:

- `None` *(default)* — display the styled HTML table.
- `'df'` — return the assembled `pandas.DataFrame` (display nothing).
- `'md'` — return a GitHub-flavored Markdown string (needs the `tabulate` package).

In [57]:
df_out = nb.table(cols='EGT', x_col='TIME', x_in=[30, 60, 90], kind='poly2', output='df')
print(type(df_out).__name__)
df_out

DataFrame


,Set,Dataset,TIME,EGT,INTERPOLATED,METHOD
0,0,Engine A,30,580.188428,Interpolated,LS2
1,0,Engine A,60,649.293211,Interpolated,LS2
2,0,Engine A,90,674.864413,Interpolated,LS2
3,1,Engine B,30,594.965480,Interpolated,LS2
4,1,Engine B,60,665.562710,Interpolated,LS2
5,1,Engine B,90,691.863527,Interpolated,LS2
6,2,Engine C,30,609.397537,Interpolated,LS2
7,2,Engine C,60,678.559667,Interpolated,LS2
8,2,Engine C,90,706.135309,Interpolated,LS2


In [58]:
print(nb.table(cols='EGT', x_col='TIME', x_in=[30, 60], kind='linear', output='md'))

Markdown output requires the 'tabulate' package (pip install tabulate).
None


## 11.4 `table_read` — interpolation in code

When you want the **numbers** rather than a display, use
`nb.table_read(slice, x_col, y_col, x_in, kind=None, fill_value='extrapolate',
bounds_error=False)`. It interpolates `y_col` at each `x_in` for every dataset in
`slice` and returns a **dict keyed by dataset index**.

> ⚠️ **Important distinction.** This low-level wrapper passes `kind` straight to
> SciPy's `interp1d`, so it only accepts *interpolation* kinds — `'linear'`,
> `'nearest'`, `'slinear'`, `'quadratic'`, `'cubic'`, `'zero'`, or an integer
> spline order. It does **not** accept regression-only specs (`'poly2'`, `'log'`,
> tuples) and will raise if you pass them. The default `kind` is each dataset's
> `reg_order` (falling back to `'linear'`), so this default also only works when
> `reg_order` is unset or itself an `interp1d`-compatible kind. **To read values
> that match a `reg_order` curve like `poly2` or `log`, use `table()` instead.**

In [59]:
# Linear interpolation of EGT at three times, for every engine
vals = nb.table_read('all', 'TIME', 'EGT', [30, 60, 90], kind='linear')
for idx, arr in vals.items():
    print(f"Set {idx}: EGT @ [30,60,90] s -> {[round(float(v), 1) for v in arr]}")

Set 0: EGT @ [30,60,90] s -> [589.3, 654.1, 661.4]
Set 1: EGT @ [30,60,90] s -> [602.1, 663.1, 685.7]
Set 2: EGT @ [30,60,90] s -> [625.3, 670.2, 702.6]


`kind` takes any `interp1d` kind — e.g. `'cubic'` for a smoother read. There's
also a **module-level** `table_read(df, x_col, y_col, x_in, …)` that works on a
bare DataFrame, no notebook required:

In [60]:
from unichart import table_read

egt_cubic = table_read(nb.sets[0].df, 'TIME', 'EGT', [30, 60, 90], kind='cubic')
print('Engine A cubic EGT @ [30,60,90]:', [round(float(v), 1) for v in egt_cubic])

# tidy up the regression state before the rest of the tutorial
nb.reg_order('all', None)
nb.reset_format()

Engine A cubic EGT @ [30,60,90]: [588.7, 657.0, 659.6]
Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


<a id="12"></a>
# 12. Saving & inspecting

| Call | Purpose |
|------|---------|
| `nb.save_png('plot.png', scale=3)` | Save the **last** figure to a high-res PNG. |
| `nb.list_sets()` | Table of datasets (title, selected, shape, query). |
| `nb.list_parms(set_number=None, search_string=None)` | List available columns/parameters. |
| `nb.summary(cols=None)` | Per-dataset summary statistics. |
| `nb.help()` | Print the class help / method overview. |

`save_png` exports whatever you most recently plotted, so plot first, then save.

In [61]:
nb.restore('all')
nb.reset_format()
nb.plot('TIME', 'EGT').show()
# nb.save_png('engine_egt.png', scale=3)   # uncomment to write the file
nb.list_parms()

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes.


Found 11 parameters in active datasets:
  - ALT                       : No description available.
  - EGT                       : No description available.
  - EGT_LIMIT                 : No description available.
  - FF                        : No description available.
  - MACH                      : No description available.
  - N1                        : No description available.
  - PHASE                     : No description available.
  - SET                       : No description available.
  - TIME                      : No description available.
  - TITLE                     : No description available.
  - TRA                       : No description available.


['ALT',
 'EGT',
 'EGT_LIMIT',
 'FF',
 'MACH',
 'N1',
 'PHASE',
 'SET',
 'TIME',
 'TITLE',
 'TRA']

In [62]:
nb.summary(['EGT', 'N1', 'FF'])

,Set,Title,Query,Variable,Count,Min,Mean,Max,Std
0,0,Engine A,-,EGT,80,444.651943,619.537527,684.381207,62.680848
1,0,Engine A,-,N1,80,39.172792,81.033115,94.192003,14.640278
2,0,Engine A,-,FF,80,975.982657,2258.388723,2695.141995,455.515533
3,1,Engine B,-,EGT,80,454.514018,635.285622,695.058924,63.999161
4,1,Engine B,-,N1,80,40.547865,82.510676,96.170393,14.711938
5,1,Engine B,-,FF,80,1005.081323,2387.503867,2809.687994,468.537734
6,2,Engine C,-,EGT,80,465.610664,650.134842,718.304354,63.184741
7,2,Engine C,-,N1,80,41.880476,84.107974,97.399560,14.643560
8,2,Engine C,-,FF,80,1196.083496,2489.879047,2967.375124,464.628957


<a id="13"></a>
# 13. Cheat sheet

```python
# --- load & select ----------------------------------------------------
nb = UnichartNotebook()
nb.load_df(df, set_idx_column='SET', set_name_column='TITLE')
nb.select([0, 2]); nb.omit(1); nb.restore('all')
nb.query('all', 'PHASE == "Cruise"')          # filter rows; '' clears

# --- the two main plots ----------------------------------------------
nb.plot('X', 'Y')                              # scatter (markers by default)
nb.plot('X', ['Y1', 'Y2'])                     # subplot per variable
nb.plot('X', 'Y', by='sets')                   # subplot per dataset
nb.plot_ymult('X', ['Y1', 'Y2', 'Y3'])         # one plot, many y-axes

# --- per-dataset formatting (persists) -------------------------------
nb.color(0, 'red'); nb.marker(0, 's'); nb.linestyle('all', '--')
nb.markersize('all', 9); nb.linewidth(0, 3); nb.alpha(0, 0.5)
nb.fill('all', False); nb.edgewidth('all', 2)
nb.hue('all', 'ALT'); nb.hue_palette('all', 'Viridis')
nb.reg_order('all', 'linear')

# --- per-variable formatting (overrides dataset) ---------------------
nb.var_format('EGT', color='red', linestyle='--')
nb.color('EGT', 'red')                          # shortcut for the same thing
nb.clear_var_format()

# --- palettes ---------------------------------------------------------
nb.color_map = [...]; nb.marker_map = [...]
nb.set_color_palette('Bold', 'all')

# --- decorations ------------------------------------------------------
nb.line('Y', 650, color='red', linestyle='--')  # h/v reference line
nb.highlight('X', (0, 20), color='gray', alpha=0.15)
nb.scale('Y', (400, 700))                        # axis limits
nb.suptitle = '...'; nb.x_label = '...'; nb.y_label = '...'
nb.set_font_sizes(suptitle='xl', legend='sm')
nb.toggle_darkmode(True)

# --- regression & tables ---------------------------------------------
nb.reg_order('all', 'poly2')                    # fit a curve (drawn on next plot)
nb.plot('X', 'Y'); nb.reg_info('all')           # equation + R²/RMSE/MAE (plot first)
nb.table(['X', 'Y'])                            # raw values table
nb.table('Y', x_col='X', x_in=[1, 2, 3], kind='poly2')   # values off the fitted curve
nb.table('Y', x_col='X', x_in=[1, 2], output='df')       # -> DataFrame ('md' for markdown)
vals = nb.table_read('all', 'X', 'Y', [1, 2, 3], kind='linear')   # -> dict of numbers

# --- other charts -----------------------------------------------------
nb.bar('PHASE', 'EGT', agg='mean')
nb.box('PHASE', 'EGT', points='all')
nb.histogram('EGT', bin_size=25)
nb.contour('MACH', 'ALT', z='EGT')

# --- save & inspect ---------------------------------------------------
nb.save_png('plot.png', scale=3)
nb.list_sets(); nb.list_parms(); nb.summary(); nb.help()
```

Happy plotting! 🛩️